In [3]:
from utils import *
import torch

In [4]:
from sdc_parser import *
from sdc_system import *
from proxy_a import *
from sdc_torch import *

--------------------------------------------------------------------------

  Local host:   cn138
  Local device: mlx5_0
--------------------------------------------------------------------------


In [5]:
device='cuda'

In [6]:
rank = 0
numranks=1

In [17]:
sy = system(1)
sy.latticeVectors,sy.symbols,sy.types,sy.coords = \
    read_coords_file("coords_10000.pdb",lib="None",verb=True)
sy.nats = len(sy.coords)

In [25]:
Rcut = 5.

In [26]:
%timeit nl = build_nlist_torch(sy.coords,sy.latticeVectors,rcut=Rcut,device=device,rank=rank,numranks=numranks,verb=True)

Building neighbor list ...
Time for copying arrays to device =  0.0009303069673478603  sec
Time for building boxneighs =  0.00013995799235999584  sec
Time for repeating coords =  0.0010706030298024416  sec
Time for distance calculation =  0.0005888290470466018  sec
Time for building neighbor list vectors =  0.0003235139884054661  sec
Time for copying nlVect to host =  0.04848645406309515  sec
Time for converting nlVect to numpy =  2.5736982934176922e-05  sec
Building neighbor list ...
Time for copying arrays to device =  0.0005658019799739122  sec
Time for building boxneighs =  0.00011945003643631935  sec
Time for repeating coords =  0.0002897439990192652  sec
Time for distance calculation =  0.0005406390409916639  sec
Time for building neighbor list vectors =  0.00036002788692712784  sec
Time for copying nlVect to host =  0.04950712400022894  sec
Time for converting nlVect to numpy =  1.6749952919781208e-05  sec
Building neighbor list ...
Time for copying arrays to device =  0.0006312

In [27]:
coord = torch.tensor(sy.coords)

In [28]:
LBox = torch.tensor(np.diagonal(sy.latticeVectors))

In [29]:
Nr_atoms = len(coord)

In [30]:
# Run on V100 GPU
%timeit nnR, nnDist, nnType = nearestneighborlist_Fortran(coord.to(device), Nr_atoms, LBox.to(device), Rcut, device)

795 ms ± 20.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
# Run on V100 GPU
nnR, nnDist, nnType = nearestneighborlist_Fortran(coord.to(device), Nr_atoms, LBox.to(device), Rcut, device)

In [32]:
nl = build_nlist_torch(sy.coords,sy.latticeVectors,5.0,device=device,rank=rank,numranks=numranks,verb=False)

In [33]:
nnType_np = nnType.cpu().numpy()

In [34]:
all_true = True
for i in range(len(nl)):
  compare_row = nl[i,1:][nl[i,1:]!=-1] == np.flip(np.sort(nnType_np[i][nnType_np[i]!=-1]))
  if not np.all(compare_row):
      print("Neighbors of atom ",i," are not the same")
      all_true = False
print("all_true =",all_true)

all_true = True


In [35]:
i = 15
print(nl[i,1:][nl[i,1:]!=-1])
print(np.flip(np.sort(nnType_np[i][nnType_np[i]!=-1])))

[4896 4823 4172 4170 4072 4071 3977 3734 3704 3605 3604 3603 3542 3541
 3540 3536 3535 3534 1403 1340 1338 1216 1139 1138 1137  349  347  346
  345  332  331  330  296  295  294  248  247  246  239  238  237   83
   82   81   68   67   66   17   16   15]
[4896 4823 4172 4170 4072 4071 3977 3734 3704 3605 3604 3603 3542 3541
 3540 3536 3535 3534 1403 1340 1338 1216 1139 1138 1137  349  347  346
  345  332  331  330  296  295  294  248  247  246  239  238  237   83
   82   81   68   67   66   17   16   15]
